## Usage

This notebook takes a set of diffraction patterns, thresholding parameters and meta data and Calibrates the camera length, real space pixel size and rotation angel 

### Parameters

 visit: str

The path to the convered processing folder (containing the hdf5 file containing the diffraction patterns)

 sample: str

The name of the sample

 timestamp: str

The timestamp of the dataset to be processed

 maskpath: str

The path to the mask to apply to the data, leave blank if no mask is required
 
 outpath: str 

The full path to the output folder (must be within the vist)

 lower: float

The lower threhold used to determine the width of the bright field disc

 upper: float

Th upper threshold used to determine the width of the bright field disc

### Dependencies

- h5py
- py4DSTEM
- matplotlib
- numpy

In [ ]:
import numpy as np
import py4DSTEM
import h5py
import os
import matplotlib.pyplot as plt

In [ ]:
print(f'visit: {visit}, type: {print(type(visit))}')
print(f'sample: {sample}, type: {print(type(sample))}')
print(f'timestamp: {timestamp}, type: {print(type(timestamp))}')

### Process the timestamp can vary based on user input:

In [ ]:
if type(timestamp) == int:
    print('timestamp was submitted with underscore')
    timestamp_proc = str(timestamp)
    timestamp_proc = timestamp_proc[:-6] + '_' + timestamp_proc[-6:]
    print(timestamp_proc)
elif type(timestamp) == str:
    timestamp_proc = timestamp.split(' ')[0] + '_' + timestamp.split(' ')[1]
    print(timestamp_proc)


### Load the meta data of the experiment

In [ ]:
metadata_path = visit + '/processing/Merlin/' + sample + '/' + timestamp_proc + '/' + timestamp_proc + '.hdf'
print(f'metadata_path: {metadata_path}')
with h5py.File(metadata_path,'r') as f:
    print('Metadata keys: %s'%f['metadata'].keys())
    step_size = f['metadata']['step_size(m)'][()]
    acc_voltage = f['metadata']['ht_value(V)'][()]
    alpha = f['metadata']['convergence_semi-angle(rad)'][()]
    nomial_CL = f['metadata']['nominal_camera_length(m)'][()]

### Load a subsection of the data

In [ ]:
data_path = visit + '/processing/Merlin/' + sample + '/' + timestamp_proc + '/' + timestamp_proc + '_data.hdf5'
print(f'data_path: {data_path}')

In [ ]:
#load diffraction patterns
f = h5py.File(data_path,'r')
dataset = py4DSTEM.DataCube(
    f['Experiments']['__unnamed__']['data'][0:64,0:64,:,:]
)
f.close()

### Load the mask for hot pixels and cross

In [ ]:
with h5py.File(maskpath, 'r') as f:
    print(f.keys())
    mask = f['data/mask'][()]
mask = mask.astype('bool')
mask = np.invert(mask)

#### Check the size of the data in order to apply the right mask area

In [ ]:
if dataset.data.shape[-1] == 515:
    dataset.data = dataset.data*mask
elif dataset.data.shape[-1] == 256:
    dataset.data = dataset.data*mask[dataset.data.shape[-2:]]

### Determine the size of the bright field disc in pixels

In [ ]:
# Calculate the mean diffraction pattern
dataset.get_dp_mean()
dataset.get_dp_max()

In [ ]:
# plot the mean diffraction pattern and vacuum probe measurement
py4DSTEM.show(
    [dataset.tree('dp_mean'),
     dataset.tree('dp_max')],
)

In [ ]:
# Estimate the radius of the bright field disc, you will have alter the thresholds
probe_radius_pixels, probe_qx0, probe_qy0 = dataset.get_probe_size(
    thresh_lower=lower,                                                                             
    thresh_upper=upper,
    plot = True
)

#### Display the radius and center of the bright field disc in terms of x and y

In [ ]:
print(f'Bright field disc radius in pixels: {probe_radius_pixels},')
print(f'Centre of the bright field disc: x={probe_qx0}, y={probe_qy0}')

#### Determine the camera length and real space pixel size

In [ ]:
mrad_per_pix = (alpha*1000)/ probe_radius_pixels
print(f'Angle subtended by a single pixel in mrad: {mrad_per_pix}')

In [ ]:
CL = 55e-6/(mrad_per_pix/1000)
print(f'Measured value of the camera length: {CL} m')

In [ ]:
print(f'The ratio between the measured camera length and the nomial camera length: {CL/nomial_CL}')

In [ ]:
wav = 1e-12*(1.23e3/np.sqrt(acc_voltage*(1+9.78e-7*acc_voltage)))
print(f'Approximate electron wavelength: {wav} m')

In [ ]:
print(f'The real space pixel size: {wav/(dataset.data.shape[-1]*mrad_per_pix/1000)} m')

### Calibrate the dataset for dpc processing

In [ ]:
dataset.calibration.set_R_pixel_size(step_size/1e-10)
dataset.calibration.set_R_pixel_units('A')
dataset.calibration.set_Q_pixel_size(mrad_per_pix)
dataset.calibration.set_Q_pixel_units('mrad')

In [ ]:
dpc = py4DSTEM.process.phase.DPC(
   datacube=dataset, 
   energy = acc_voltage,
).preprocess()

In [ ]:
dpc.reconstruct(
    max_iter=8,
    store_iterations=True,
    reset=True,
).visualize(
    iterations_grid='auto',
    figsize=(12,10)
);